# 03 — XCom & TaskFlow API

XCom = cross-task communication. Stored in Airflow metadata DB.

**Rule #1: Never push DataFrames.** XCom is for small values only.
**Rule #2: For large data — push the S3 path, not the data.**

## 1. Explicit push / pull

In [ ]:
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator

def upstream(**kwargs):
    ti = kwargs["ti"]
    ti.xcom_push(key="row_count", value=42_000)
    ti.xcom_push(key="s3_path",   value="s3://bucket/output/2024-01-15/data.parquet")
    print("Pushed row_count and s3_path to XCom")

def downstream(**kwargs):
    ti = kwargs["ti"]
    count    = ti.xcom_pull(task_ids="upstream_task", key="row_count")
    s3_path  = ti.xcom_pull(task_ids="upstream_task", key="s3_path")
    print(f"Got {count} rows from {s3_path}")

with DAG("xcom_explicit", start_date=datetime(2024,1,1), catchup=False) as dag:
    t1 = PythonOperator(task_id="upstream_task",   python_callable=upstream)
    t2 = PythonOperator(task_id="downstream_task", python_callable=downstream)
    t1 >> t2

print("XCom explicit DAG built")

## 2. Implicit push (return value)

In [ ]:
def compute():   # return value auto-pushed as key="return_value"
    return {"status": "ok", "rows": 42_000, "path": "s3://bucket/out.parquet"}

def consume(**kwargs):
    result = kwargs["ti"].xcom_pull(task_ids="compute_task")  # pulls return_value
    print(f"rows={result['rows']}, path={result['path']}")

with DAG("xcom_implicit", start_date=datetime(2024,1,1), catchup=False) as dag:
    t1 = PythonOperator(task_id="compute_task", python_callable=compute)
    t2 = PythonOperator(task_id="consume_task", python_callable=consume)
    t1 >> t2

print("XCom implicit DAG built")

## 3. TaskFlow API — cleanest XCom pattern

`@task` decorator wires XCom automatically. Return values flow as function arguments.

In [ ]:
from airflow.decorators import dag, task

@dag(schedule="@daily", start_date=datetime(2024, 1, 1), catchup=False, tags=["taskflow"])
def asin_pipeline():

    @task
    def extract(ds=None) -> dict:
        print(f"Extracting for {ds}")
        return {"s3_path": f"s3://bucket/raw/{ds}/data.parquet", "row_count": 50_000}

    @task
    def validate(data: dict) -> dict:
        print(f"Validating {data['row_count']} rows from {data['s3_path']}")
        return {**data, "valid": True}

    @task
    def transform(data: dict) -> str:
        print(f"Transforming {data['row_count']} rows")
        return "s3://bucket/processed/data.parquet"

    @task
    def load(output_path: str):
        print(f"Loading from {output_path} to Redshift")

    # XCom is wired automatically — return values flow as args
    raw      = extract()
    verified = validate(raw)
    output   = transform(verified)
    load(output)

pipeline = asin_pipeline()
print("TaskFlow DAG created:", pipeline.dag_id)

## 4. XCom rules — what interviewers ask

| Rule | Why |
|---|---|
| Never push DataFrames | XCom stored in metadata DB (Postgres/SQLite). Large payloads crash it |
| Push S3 path instead | Decouple data transfer from orchestration |
| MWAA practical limit | Keep under 48KB |
| XCom is DAG-scoped | Can't XCom across DAGs — use shared S3 path + ExternalTaskSensor |

**Interview answer on large data:**
> *"I never push DataFrames via XCom — XCom is stored in the Airflow metadata DB and large payloads corrupt it. In A2H, tasks pushed the S3 output path as XCom and downstream tasks read directly from S3. Same pattern I'd use anywhere."*

## 5. BranchPythonOperator with XCom

In [ ]:
from airflow.operators.python import BranchPythonOperator
from airflow.operators.empty import EmptyOperator

def get_count():   return 75_000

def decide(**kwargs):
    count = kwargs["ti"].xcom_pull(task_ids="count")
    return "full" if count > 50_000 else "sample"

with DAG("branch_xcom", start_date=datetime(2024,1,1), catchup=False) as dag:
    count  = PythonOperator(task_id="count",  python_callable=get_count)
    branch = BranchPythonOperator(task_id="branch", python_callable=decide)
    full   = EmptyOperator(task_id="full")
    sample = EmptyOperator(task_id="sample")
    done   = EmptyOperator(task_id="done", trigger_rule="none_failed_min_one_success")

    count >> branch >> [full, sample] >> done

print("Branch + XCom DAG built")